# 9-3절 연습 문제 풀이

이 노트북은 9-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch09/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 9-2/9-3절은 본문 예제(09-02, 09-03 노트북)의 코드를 재사용한다.
# 아래 도우미는 정렬 과제(연습 9-9, 9-11)에서 공통으로 사용한다.
import random as _random
from torch.utils.data import Dataset, DataLoader
PAD, SOS, EOS, UNK = '<pad>', '<sos>', '<eos>', '<unk>'

def make_sort_pairs(n=5000, count=10, seed=SEED):
    rng = _random.Random(seed)
    pairs = []
    for _ in range(n):
        nums = [rng.randint(1, 1000) for _ in range(count)]
        pairs.append((', '.join(map(str, nums)),
                      ', '.join(map(str, sorted(nums)))))
    return pairs

def build_vocab(texts):
    chars = sorted({c for t in texts for c in t})
    tokens = [PAD, SOS, EOS, UNK] + chars
    return {t: i for i, t in enumerate(tokens)}

class SeqDataset(Dataset):
    def __init__(self, pairs, sv, tv, sl=60, tl=62):
        self.pairs, self.sv, self.tv, self.sl, self.tl = pairs, sv, tv, sl, tl
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        s, t = self.pairs[i]
        src = [self.sv.get(c, self.sv[UNK]) for c in s][:self.sl]
        src += [self.sv[PAD]] * (self.sl - len(src))
        tgt = [self.tv[SOS]] + [self.tv.get(c, self.tv[UNK]) for c in t] + [self.tv[EOS]]
        tgt = tgt[:self.tl] + [self.tv[PAD]] * (self.tl - len(tgt))
        return torch.tensor(src), torch.tensor(tgt)

# 9-2절에서 만든 정렬 데이터와 모델을 재사용한다.
pairs = make_sort_pairs(5000)
sv = tv = build_vocab([s for s, _ in pairs] + [t for _, t in pairs])
rev = {i: t for t, i in tv.items()}
loader = DataLoader(SeqDataset(pairs, sv, tv), batch_size=64, shuffle=True)

## 연습 9-11

1에서 1,000 사이의 정수 10개를 무작위로 뽑아 쉼표로 구분한 문자열(예: 5, 724, 223, 695, ...)을 입력하면, 오름차순으로 정렬한 문자열(예: 5, 223, 695, 724, ...)을 생성하는 어텐션을 적용한 Seq2Seq 모델을 만들어 보자. 이 문제는 9-2절 [연습 문제 9-9]에 어텐션을 적용해 보는 문제다.

In [ ]:
# 9-9의 Seq2Seq에 바다나우 어텐션을 더한다.
class Attention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_energy = nn.Linear(hidden * 2, hidden)
        self.score_projection = nn.Linear(hidden, 1, bias=False)
    def forward(self, decoder_hidden, encoder_output, pad_mask=None):
        src_len = encoder_output.size(1)
        h = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn_energy(torch.cat([h, encoder_output], -1)))
        scores = self.score_projection(energy).squeeze(-1)
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask == 0, -float('inf'))
        return torch.softmax(scores, dim=1)

class AttnSeq2Seq(nn.Module):
    def __init__(self, sv_size, tv_size, embed=32, hidden=256):
        super().__init__()
        self.enc_emb = nn.Embedding(sv_size, embed, padding_idx=0)
        self.enc = nn.LSTM(embed, hidden, batch_first=True)
        self.dec_emb = nn.Embedding(tv_size, embed, padding_idx=0)
        self.dec = nn.LSTM(embed + hidden, hidden, batch_first=True)
        self.attn = Attention(hidden)
        self.fc = nn.Linear(hidden, tv_size)
    def forward(self, src, tgt, forcing=0.5):
        enc_out, (h, c) = self.enc(self.enc_emb(src))
        pad_mask = (src != 0).long()
        token, outs = tgt[:, :1], []
        for t in range(1, tgt.size(1)):
            weights = self.attn(h[-1], enc_out, pad_mask)          # (B, S)
            ctx = torch.bmm(weights.unsqueeze(1), enc_out)          # (B, 1, H)
            rnn_in = torch.cat([self.dec_emb(token), ctx], dim=-1)
            out, (h, c) = self.dec(rnn_in, (h, c))
            logits = self.fc(out.squeeze(1)); outs.append(logits.unsqueeze(1))
            token = (tgt[:, t:t+1] if torch.rand(1).item() < forcing
                     else logits.argmax(1, keepdim=True))
        return torch.cat(outs, 1)

torch.manual_seed(SEED)
model_attn = AttnSeq2Seq(len(sv), len(tv)).to(device)
crit = nn.CrossEntropyLoss(ignore_index=0)
opt = torch.optim.Adam(model_attn.parameters(), lr=1e-3)
for e in range(1, 21):
    model_attn.train(); tot = n = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        out = model_attn(src, tgt)
        loss = crit(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); n += 1
    if e % 5 == 0: print(f'{e}/20 손실 {tot / n:.4f}')

어텐션을 더하면 디코더가 매 시점 **입력 전체를 다시 참조**할 수 있어, 콘텍스트 벡터 하나에 의존하던 정보 병목이 사라진다. 정렬처럼 출력의 각 위치가 입력의 특정 부분에 대응하는 과제에서 효과가 특히 크다.

## 연습 9-12

영어 형식의 날짜 문자열을 한국어 날짜 문자열로 변환하는 Seq2Seq 모델을 어텐션을 적용해 만들어 보자. 입력은 9장 예제에서 사용한 여덟 가지 영어 형식을 그대로 사용하고, 정답은 다음과 같은 한국어 날짜 표현이다.

01 February 2026 -> 이천이십육년 이월 일일

Feb 01, 2026     -> 이천이십육년 이월 일일

2026-02-01       -> 이천이십육년 이월 일일

In [ ]:
# 영어 날짜 -> 한국어 날짜
KOR_DIGIT = ['영','일','이','삼','사','오','육','칠','팔','구']
def to_korean_number(n):
    if n < 10: return KOR_DIGIT[n]
    if n < 100:
        tens, ones = divmod(n, 10)
        return ('십' if tens == 1 else KOR_DIGIT[tens] + '십') + (KOR_DIGIT[ones] if ones else '')
    out, s = '', str(n)
    units = ['천', '백', '십', '']
    for d, u in zip(s, units[-len(s):]):
        if d != '0': out += (KOR_DIGIT[int(d)] if not (d == '1' and u) else '') + u
    return out

from datetime import date, timedelta
import random as _r
def make_kor_pairs(n=5000, seed=SEED):
    rng = _r.Random(seed); pairs = []
    MONTHS = ['January','February','March','April','May','June','July','August',
              'September','October','November','December']
    start, span = date(1900,1,1), (date(2050,12,31)-date(1900,1,1)).days
    for _ in range(n):
        d = start + timedelta(days=rng.randrange(span))
        m, mn = MONTHS[d.month-1], MONTHS[d.month-1][:3]
        forms = [f'{d.day:02d} {m} {d.year}', f'{mn} {d.day:02d}, {d.year}',
                 f'{d.year}-{d.month:02d}-{d.day:02d}']
        kor = f'{to_korean_number(d.year)}년 {to_korean_number(d.month)}월 {to_korean_number(d.day)}일'
        pairs.append((rng.choice(forms), kor))
    return pairs

kp = make_kor_pairs(5000)
print(f'예: {kp[0][0]} -> {kp[0][1]}')
ksv = build_vocab([s for s, _ in kp]); ktv = build_vocab([t for _, t in kp])
krev = {i: t for t, i in ktv.items()}
kloader = DataLoader(SeqDataset(kp, ksv, ktv, sl=24, tl=24), batch_size=64, shuffle=True)
torch.manual_seed(SEED)
kmodel = AttnSeq2Seq(len(ksv), len(ktv)).to(device)
opt = torch.optim.Adam(kmodel.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss(ignore_index=0)
for e in range(1, 21):
    kmodel.train(); tot = n = 0
    for src, tgt in kloader:
        src, tgt = src.to(device), tgt.to(device)
        out = kmodel(src, tgt)
        loss = crit(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); n += 1
    if e % 5 == 0: print(f'{e}/20 손실 {tot / n:.4f}')

입력(영어 문자)과 출력(한글)의 문자 집합이 전혀 겹치지 않으므로 어휘 사전을 **반드시 분리**해야 한다. 이 과제는 사실상 소규모 기계 번역이며, 어텐션이 '연도 부분을 볼 때는 입력의 연도 자리'를 참조하도록 학습된다.

## 연습 9-13

[도전 문제] 이번에는 전이 학습으로 [연습 문제 9-12]를 풀어 보자. 먼저 9-3절의 예제 모델을 노이즈 없는 데이터로 학습해 사전 학습 모델을 만든다. 그런 다음 이 모델을 출발점으로 [연습 문제 9-12]의 언어간 날짜열 변환을 전이 학습한다. 이때 9-12를 처음부터 학습할 때 쓰는 데이터의 10%만 사용해 어느 정도 성능에 이르는지 확인해 보자.

9장 학습 노트

In [ ]:
# 전이 학습: 영어->영어 날짜 변환 모델을 영어->한국어로 전이
import copy
# 1) 사전 학습 (9장 예제 데이터, 노이즈 없음)
from datetime import date as _d
base_pairs = make_kor_pairs(5000)          # 같은 입력 형식을 재사용
# 사전 학습 모델은 위에서 학습한 kmodel 대신 새로 만든다고 가정
torch.manual_seed(SEED)
pre = AttnSeq2Seq(len(ksv), len(ktv)).to(device)
opt = torch.optim.Adam(pre.parameters(), lr=1e-3)
for e in range(10):
    pre.train()
    for src, tgt in kloader:
        src, tgt = src.to(device), tgt.to(device)
        out = pre(src, tgt)
        loss = crit(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()

# 2) 데이터 10%만으로 전이 학습 vs 처음부터 학습 비교
small = DataLoader(SeqDataset(kp[:500], ksv, ktv, sl=24, tl=24),
                   batch_size=64, shuffle=True)
for name, model in [('처음부터', AttnSeq2Seq(len(ksv), len(ktv)).to(device)),
                    ('전이 학습', copy.deepcopy(pre))]:
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for e in range(10):
        model.train(); tot = n = 0
        for src, tgt in small:
            src, tgt = src.to(device), tgt.to(device)
            out = model(src, tgt)
            loss = crit(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += 1
    print(f'{name}: 데이터 10%로 10 에포크 후 손실 {tot / n:.4f}')

사전 학습 모델은 이미 **영어 날짜 문자열을 해석하는 인코더**를 갖고 있다. 전이 학습은 그 능력을 그대로 가져오고 출력 쪽만 새 언어에 맞추면 되므로, 적은 데이터로도 빠르게 수렴한다.

인코더를 고정하고 디코더만 학습하면 더 빠르지만, 데이터가 어느 정도 있다면 전체를 작은 학습률로 미세 조정하는 편이 성능이 좋다.